In [ ]:
import os
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("SparkExample") \
    .config("spark.jars.packages",
        "org.apache.hadoop:hadoop-aws:3.3.4,"
        "com.amazonaws:aws-java-sdk-bundle:1.12.262,"
        "ru.yandex.clickhouse:clickhouse-jdbc:0.3.2,"
        "org.postgresql:postgresql:42.5.0,"
        "org.apache.spark:spark-sql-kafka-0-10_2.12:3.3.0",
        ) \
    .getOrCreate()


hadoop_conf = spark._jsc.hadoopConfiguration()
hadoop_conf.set("fs.s3a.access.key", os.getenv("MINIO_ROOT_USER"))
hadoop_conf.set("fs.s3a.secret.key", os.getenv("MINIO_ROOT_PASSWORD"))
hadoop_conf.set("fs.s3a.endpoint", "http://minio:9000")
hadoop_conf.set("fs.s3a.connection.ssl.enabled", "false")
hadoop_conf.set("fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
hadoop_conf.set("fs.s3a.path.style.access", "true")

In [ ]:
# ⬇️ Параметры подключения к PostgreSQL public.shops 
jdbc_url = "jdbc:postgresql://postgres_source:5432/source" 
table_name = "public.shops" 
db_user = os.getenv("POSTGRES_USER") 
db_password = os.getenv("POSTGRES_PASSWORD") 
shops_df = spark.read \
                    .format("jdbc") \
                    .option("url", jdbc_url) \
                    .option("user", db_user) \
                    .option("password", db_password) \
                    .option("dbtable", table_name) \
                    .option("fetchsize", 1000) \
                    .option("driver", "org.postgresql.Driver") \
                    .load() 
shops_df.show()

In [ ]:
# ⬇️ Параметры подключения к PostgreSQL public.shop_timezone 
jdbc_url = "jdbc:postgresql://postgres_source:5432/source" 
table_name = "public.shop_timezone" 
db_user = os.getenv("POSTGRES_USER") 
db_password = os.getenv("POSTGRES_PASSWORD") 
shop_timezone_df = spark.read \
				.format("jdbc") \
				.option("url", jdbc_url) \
				.option("user", db_user) \
				.option("password", db_password) \
				.option("dbtable", table_name) \
				.option("fetchsize", 1000) \
				.option("driver", "org.postgresql.Driver") \
				.load() 

shop_timezone_df.show()

In [ ]:
# Вариант решения на Spark SQL

# Регистрируем DataFrame-ы как временные вьюхи 
shops_df.createOrReplaceTempView("shops") 
shop_timezone_df.createOrReplaceTempView("shop_timezone")

# Выполняем SQL запрос для трансформации 
spark.sql(""" 
WITH name AS (SELECT cast(st_id as INT) AS st_id,
                     shop_name
                FROM shops),
 timezone AS (SELECT try_cast(plant as INT) AS st_id,
                     cast(CASE WHEN time_zone = '' OR time_zone IS NULL THEN 3
                               ELSE substr(time_zone, 4) END AS INT) AS tz_code 
                FROM shop_timezone
               WHERE try_cast(plant as INT) IS NOT NULL),
        t AS (SELECT st_id, 
                     shop_name, 
                     tz_code
                FROM name JOIN timezone USING(st_id))
SELECT st_id,
       shop_name,
       max(tz_code) AS tz_code
  FROM t
 GROUP BY st_id, shop_name
 ORDER BY st_id
""").show()

In [ ]:
# Вариант решения на PySpark

from pyspark.sql import functions as f
from pyspark.sql.types import IntegerType

# 1. Справочник имен
names = (
    shops_df
    .select(
        f.col("st_id").try_cast(IntegerType()).alias("st_id"),
        "shop_name"
    )
    .where("try_cast(st_id as INT) IS NOT NULL")
)

# 2. Справочник таймзон
timezones = (
    shop_timezone_df
    .where("try_cast(plant as INT) IS NOT NULL")
    .select(
        f.col("plant").try_cast(IntegerType()).alias("st_id"),
        f.when(
            (f.col("time_zone") == "") | (f.col("time_zone").isNull()), 
            3
        ).otherwise(
            f.substring(f.col("time_zone"), 4, 10).try_cast(IntegerType())
        ).alias("tz_code")
    )
)

# 3. Финальная сборка
result_df = (
    names
    .join(timezones, on="st_id", how="inner")
    .groupBy("st_id", "shop_name")
    .agg(f.max("tz_code").alias("tz_code"))
    .orderBy("st_id")
)

result_df.show()

In [ ]:
spark.stop()